In [1]:
import sys
sys.path.append('..')

from hybrid_rag.pipeline import HybridRAGPipeline

hybrid = HybridRAGPipeline()

🔧 Initialising Hybrid RAG Pipeline...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ ChromaDB loaded — 2289 child vectors
✅ BM25 loaded — 7913 children
Mistral client ready - model: mistral-medium-latest
✅ Hybrid RAG ready — 2289 vectors | 7913 BM25 children | 2010 parents



In [2]:
from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

TEST_QUESTIONS = [
    "What was NVIDIA's total revenue for the most recent fiscal year?",
    "What percentage of NVIDIA's revenue came from data center products?",
    "What was Amazon Web Services revenue for the most recent fiscal year?",
    "What is Microsoft's stated strategy for artificial intelligence investments?",
    "What risks related to content licensing did Netflix identify?",
]

for q in TEST_QUESTIONS:
    print(f"\n{'#'*60}")
    print(f"  Q: {q}")
    print(f"{'#'*60}")

    print("Running Vector...")
    r_vec = vec.ask(q)

    print("Running Vectorless...")
    r_vl = vl.ask(q)

    print("Running Hybrid...")
    r_hybrid = hybrid.ask(q)

    print(f"\n[VECTOR]     {r_vec['answer'][:200]}")
    print(f"  ↳ time: {r_vec['total_time']}s")

    print(f"\n[BM25]       {r_vl['answer'][:200]}")
    print(f"  ↳ time: {r_vl['total_time']}s")

    print(f"\n[HYBRID]     {r_hybrid['answer'][:200]}")
    print(f"  ↳ time: {r_hybrid['total_time']}s  "
          f"(vector: {r_hybrid['vector_latency']}s  "
          f"bm25: {r_hybrid['bm25_latency']}s  "
          f"rerank: {r_hybrid['rerank_latency']}s)")
    print(f"  ↳ candidates: vector={r_hybrid['vector_candidates']} "
          f"bm25={r_hybrid['bm25_candidates']} "
          f"fused={r_hybrid['fused_candidates']} "
          f"final={len(r_hybrid['retrieved'])}")

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 2289 child vectors
Mistral client already initialised - reusing
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents


############################################################
  Q: What was NVIDIA's total revenue for the most recent fiscal year?
############################################################
Running Vector...
🔁 Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker ready
Running Vectorless...
Running Hybrid...

[VECTOR]     For **NVIDIA**, the total revenue for the most recent fiscal year (ended **January 25, 2026**) was **$215,938 million** (or **$215.94 billion**). This figure is sourced from the Consolidated Statement
  ↳ time: 12.9567s

[BM25]       For **NVIDIA**, the total revenue for the most recent fiscal year (ended **January 25, 2026**) was **$215.9 billion**, reflecting a **65% increase** from the prior year.
  ↳ time: 1.7423s

[HYBRID]     For **NVIDIA**, the total revenue for the most recent fiscal year (ended **January 25, 2026**) was **$215,938 million** (or **$215.94 billion**). This figure is found in the Consolidated Statements of
  ↳ time: 2.7954s  (vector: 0.0862s  bm25: 0.0362s  rerank: 0.2244s)
  ↳ candidates: vector=15 bm25=15 fused=15 final=5

############################################################
  Q: What percentage of NVIDIA's revenue came from data center products?
#####################################

In [3]:
results = vec.collection.get(
    include=["metadatas"]
)

companies = set(
    m["company"]
    for m in results["metadatas"]
)

print(companies)
print(len(results["metadatas"]))

{'NETFLIX', 'NVIDIA'}
2289


In [8]:
q = "What risks related to content licensing did Netflix identify?"

ret = vec.ask(q)

print("Retrieved Chunks:", len(ret["retrieved"]))

for i, chunk in enumerate(ret["retrieved"]):
    print("\n" + "="*80)
    print(f"Chunk {i+1}")
    print("="*80)
    print(chunk["text"][:2000])

Retrieved Chunks: 4

Chunk 1
EXHIBIT 19.1
Insider Trading Policy
In order to take an active role in the prevention of insider trading violations by executive officers, directors, employees and other related individuals of Netflix,
Inc. (the “Company”) and its subsidiaries, the Company has adopted this Insider Trading Policy (the “Policy”).
Statement of Intent
The Company opposes the misuse of material nonpublic information in the trading of securities and it is the intent of this Policy to implement procedures
designed to prevent trading based on material nonpublic information regarding the Company, including any of its subsidiaries. The Company also wishes to
discourage certain trading in its securities by its executive officers, directors, employees and other related individuals that may be contrary to the interests of
our shareholders. The term "executive officer" herein shall have the same meaning as the term “officer” as defined under Rule 16a-1(f) of the Securities

Chunk 2
EXHIB

In [9]:
ret = vec.retriever.retrieve(
    "What risks related to content licensing did Netflix identify?"
)

AttributeError: 'VectorRAGPipeline' object has no attribute 'retriever'

In [10]:
from vector_rag.retriever import retrieve

ret = retrieve(
    "What risks related to content licensing did Netflix identify?",
    vec.collection,
    vec.parent_lookup,
)

print(ret)

{'chunks': [{'text': 'EXHIBIT 19.1\nInsider Trading Policy\nIn order to take an active role in the prevention of insider trading violations by executive officers, directors, employees and other related individuals of Netflix,\nInc. (the “Company”) and its subsidiaries, the Company has adopted this Insider Trading Policy (the “Policy”).\nStatement of Intent\nThe Company opposes the misuse of material nonpublic information in the trading of securities and it is the intent of this Policy to implement procedures\ndesigned to prevent trading based on material nonpublic information regarding the Company, including any of its subsidiaries. The Company also wishes to\ndiscourage certain trading in its securities by its executive officers, directors, employees and other related individuals that may be contrary to the interests of\nour shareholders. The term "executive officer" herein shall have the same meaning as the term “officer” as defined under Rule 16a-1(f) of the Securities', 'metadata':

In [11]:
from collections import Counter

results = vec.collection.get(include=["metadatas"])

print(Counter(
    m["company"]
    for m in results["metadatas"]
))


Counter({'NVIDIA': 1914, 'NETFLIX': 375})


In [12]:
import json

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    data = json.load(f)

print("Parents:", len(data["parents"]))
print("Children:", len(data["children"]))

companies = {}

for c in data["children"]:
    company = c["company"]
    companies[company] = companies.get(company, 0) + 1

print(companies)

Parents: 2010
Children: 7913
{'AMAZON': 1468, 'MICROSOFT': 2242, 'NETFLIX': 2289, 'NVIDIA': 1914}


In [13]:
import json

with open("../data/processed/chunks.json", encoding="utf-8") as f:
    data = json.load(f)

sources = {}

for c in data["children"]:
    company = c["company"]

    if company not in sources:
        sources[company] = set()

    sources[company].add(c["source"])

for company, s in sources.items():
    print(company)
    print(list(s)[:10])
    print()

AMAZON
['amazon_10k.pdf']

MICROSOFT
['microsoft_10k.pdf']

NETFLIX
['netflix_10k.pdf']

NVIDIA
['nvidia_10k.pdf']



In [14]:
results = vec.collection.get(include=["metadatas"])

sources = {}

for m in results["metadatas"]:
    company = m["company"]

    if company not in sources:
        sources[company] = set()

    sources[company].add(m["source"])

for company, s in sources.items():
    print(company)
    print(list(s))

NVIDIA
['nvidia_10k.pdf']
NETFLIX
['netflix_10k.pdf']


In [15]:
import config
print(config.CHROMA_PERSIST_DIR)

C:\Users\nagal\Documents\AI\rag-benchmark\vector_rag\chroma_db


In [16]:
import shutil

shutil.rmtree(
    r"C:\Users\nagal\Documents\AI\rag-benchmark\vector_rag\chroma_db"
)

PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\nagal\\Documents\\AI\\rag-benchmark\\vector_rag\\chroma_db\\64b2ef66-4e34-4d3d-b9a7-21e2fa293469\\data_level0.bin'